In [2]:
pip install nltk

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 989.0 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 1.0 MB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.6 MB 1.1 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 1.2 MB/s  0:00:01

   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import nltk
import numpy as np
import pandas as pd
import string
import joblib

from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [4]:
df = pd.read_csv('laptop_scenarios_story_text.csv')


In [5]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    tokens = nltk.word_tokenize(str(text))           # tokenisation
    tokens = [w.lower() for w in tokens]              # lowercasing
    tokens = [w for w in tokens if w not in stop_words]   # stop word removal
    tokens = [w for w in tokens if w not in string.punctuation]  # punctuation
    tokens = [stemmer.stem(w) for w in tokens]        # stemming
    return ' '.join(tokens)

df['clean_story'] = df['story_text'].apply(preprocess)

print("Preprocessing complete.")
print("Shape after preprocessing:", df.shape)
df[['story_text', 'clean_story']].head(3)

Preprocessing complete.
Shape after preprocessing: (19545, 7)


,story_text,clean_story
0,My main problem with this Apple Ultrabook is s...,main problem appl ultrabook simpl student keep...
1,"For anyone curious, this Apple Ultrabook has b...",anyon curiou appl ultrabook machin student use...
2,I would describe my Apple Ultrabook like this:...,would describ appl ultrabook like look capabl ...


In [6]:
vectorizer = TfidfVectorizer(max_features=3500)
tfidf_matrix = vectorizer.fit_transform(df['clean_story'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Sample feature names: {vectorizer.get_feature_names_out()[:15]}")

TF-IDF matrix shape: (19545, 602)
Sample feature names: ['00kg' '02kg' '03kg' '04kg' '05kg' '06kg' '07kg' '08kg' '09kg' '0ghz'
 '10' '1024gb' '10kg' '11' '11kg']


In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(tfidf_matrix.toarray())

pca = PCA(n_components=0.65, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA components retained: {X_pca.shape[1]}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.4f}")

PCA components retained: 74
Total variance explained: 0.6516


In [8]:
kmeans = KMeans(n_clusters=3, init='k-means++', random_state=42)
df['Cluster'] = kmeans.fit_predict(X_pca)

print("KMeans clustering complete.")
print("\nCluster distribution:")
print(df['Cluster'].value_counts().sort_index())

KMeans clustering complete.

Cluster distribution:
Cluster
0     1656
1    12413
2     5476
Name: count, dtype: int64


In [9]:
feature_names = vectorizer.get_feature_names_out()
tfidf_array = tfidf_matrix.toarray()

def get_top_terms(cluster_id, top_n=10):
    indices = df[df['Cluster'] == cluster_id].index
    cluster_tfidf = tfidf_array[indices].mean(axis=0)
    top_indices = cluster_tfidf.argsort()[::-1][:top_n]
    return [feature_names[i] for i in top_indices]

print("Top terms per cluster:")
for c in sorted(df['Cluster'].unique()):
    print(f"  Cluster {c}: {get_top_terms(c)}")

Top terms per cluster:
  Cluster 0: ['care', 'strict', 'one', 'mean', 'lot', 'hurt', 'decis', 'issu', 'make', 'intel']
  Cluster 1: ['graphic', 'feel', 'intel', 'integr', 'machin', '4gb', 'use', '256gb', '8gb', 'ram']
  Cluster 2: ['offic', 'still', 'work', 'want', 'feel', 'rather', 'meet', 'mostli', 'comfort', 'notic']


In [10]:
persona_cards = {}

for c in sorted(df['Cluster'].unique()):
    cluster_data = df[df['Cluster'] == c]

    top_terms = get_top_terms(c, top_n=10)

    dominant_profile  = cluster_data['user_profile'].value_counts().idxmax()
    upgrade_tendency  = cluster_data['upgrade_first'].value_counts().idxmax()
    dominant_gain     = cluster_data['gain_class'].value_counts().idxmax()
    dominant_budget   = cluster_data['budget_class'].value_counts().idxmax()
    cluster_size      = len(cluster_data)

    label = f"{dominant_profile.title()} Users — {upgrade_tendency} Upgrade"

    upgrade_pct = round(
        cluster_data['upgrade_first'].value_counts(normalize=True).iloc[0] * 100, 1
    )
    gain_pct = round(
        cluster_data['gain_class'].value_counts(normalize=True).iloc[0] * 100, 1
    )
    interpretation = (
        f"This cluster contains {cluster_size:,} stories. "
        f"The most common user profile is '{dominant_profile}' ({dominant_budget} budget). "
        f"{upgrade_pct}% of cases recommend a '{upgrade_tendency}' as the first upgrade. "
        f"{gain_pct}% of cases fall into the '{dominant_gain}' gain class."
    )

    persona_cards[int(c)] = {
        "label":            label,
        "top_terms":        top_terms,
        "dominant_profile": dominant_profile,
        "upgrade_tendency": upgrade_tendency,
        "gain":             dominant_gain,
        "interpretation":   interpretation,
    }

print("CLUSTER PERSONA CARDS")
for cid, card in persona_cards.items():
    print(f"\nCluster {cid}")
    print(f"  Label            : {card['label']}")
    print(f"  Dominant Profile : {card['dominant_profile']}")
    print(f"  Upgrade Tendency : {card['upgrade_tendency']}")
    print(f"  Gain Class       : {card['gain']}")
    print(f"  Top Terms        : {card['top_terms'][:6]}")
    print(f"  Interpretation   : {card['interpretation']}")


CLUSTER PERSONA CARDS

Cluster 0
  Label            : Office Users — RAM Upgrade
  Dominant Profile : office
  Upgrade Tendency : RAM
  Gain Class       : Medium
  Top Terms        : ['care', 'strict', 'one', 'mean', 'lot', 'hurt']
  Interpretation   : This cluster contains 1,656 stories. The most common user profile is 'office' (low budget). 50.2% of cases recommend a 'RAM' as the first upgrade. 47.0% of cases fall into the 'Medium' gain class.

Cluster 1
  Label            : Gamer Users — RAM Upgrade
  Dominant Profile : gamer
  Upgrade Tendency : RAM
  Gain Class       : Medium
  Top Terms        : ['graphic', 'feel', 'intel', 'integr', 'machin', '4gb']
  Interpretation   : This cluster contains 12,413 stories. The most common user profile is 'gamer' (high budget). 54.4% of cases recommend a 'RAM' as the first upgrade. 50.5% of cases fall into the 'Medium' gain class.

Cluster 2
  Label            : Office Users — No Upgrade Needed Upgrade
  Dominant Profile : office
  Upgrade Tende

In [11]:
model_data = {
    # Pipeline objects
    "vectorizer":    vectorizer,
    "scaler":        scaler,
    "pca":           pca,
    "kmeans":        kmeans,

    # Pre-computed matrix
    "tfidf_matrix":  tfidf_matrix,
    "X_pca":         X_pca,

    # Dataset with cluster labels
    "dataframe":     df,

    # Cluster persona cards
    "persona_cards": persona_cards,

    # Graph settings
    "graph_threshold": 0.55,
    "graph_max_nodes": 15,
}

joblib.dump(model_data, "model.joblib")

verify = joblib.load("model.joblib")
print("\nVerification — keys in saved file:")
for k, v in verify.items():
    print(f"  {k:<20} → {type(v).__name__}")


Verification — keys in saved file:
  vectorizer           → TfidfVectorizer
  scaler               → StandardScaler
  pca                  → PCA
  kmeans               → KMeans
  tfidf_matrix         → csr_matrix
  X_pca                → ndarray
  dataframe            → DataFrame
  persona_cards        → dict
  graph_threshold      → float
  graph_max_nodes      → int
